# 01 — Why Distributed Processing

Real, timed, locally-run comparison: single-threaded Python loop vs `multiprocessing.Pool`
vs (forward reference) PySpark — plus small real demonstrations of shuffle cost and
fault-tolerance-by-recomputation. Every number below is measured on this machine, not estimated.

In [1]:
import time
import math
import random
import sys
import multiprocessing as mp

random.seed(42)
print("CPU count on this machine:", mp.cpu_count())

CPU count on this machine: 16


## Hypothesis (stated before measuring)

1. For a CPU-bound per-row transform over ~300K rows, `multiprocessing.Pool` will beat the
   single-threaded loop, and the speedup will grow with worker count but flatten out well
   before 16x (process-spawn + IPC overhead, plus this machine's real core count).
2. For a *cheap* per-row transform (work per row too small), `multiprocessing.Pool` will be
   **slower** than the single-threaded loop — the fixed overhead of spawning processes and
   pickling data back and forth dominates.
3. Neither single-threaded nor `multiprocessing` solves the *shuffle* problem (redistributing
   data across partitions for a groupBy/join) — that requires the distributed engine PySpark
   provides (Task 2), because `multiprocessing.Pool` still holds all data in one machine's RAM.

## Generate synthetic data locally (fixed seed, no download)

In [2]:
N_ROWS = 300_000
data = [random.uniform(1, 1000) for _ in range(N_ROWS)]
print(f"Generated {N_ROWS:,} synthetic rows. First 5: {data[:5]}")

Generated 300,000 synthetic rows. First 5: [639.7873716594258, 25.98574446744427, 275.7542890507501, 223.98752741067392, 736.7347429498484]


## The embarrassingly-parallel toy computation

A CPU-bound per-row transform (a stand-in for e.g. a feature computation applied independently
to every row) — no row depends on any other row, so it is "embarrassingly parallel": it can be
split across workers with zero coordination between them.

In [3]:
def cpu_bound_transform(x):
    """A synthetic CPU-bound per-row computation. No shared state, no I/O."""
    total = 0.0
    for i in range(1, 150):
        total += math.sin(x * i) ** 2 + math.log(x + i)
    return total

## 1. Single-threaded Python loop

In [4]:
t0 = time.perf_counter()
result_single = [cpu_bound_transform(x) for x in data]
t_single = time.perf_counter() - t0
print(f"Single-threaded: {t_single:.3f}s  (checksum={sum(result_single):.2f})")

Single-threaded: 4.331s  (checksum=298113118.09)


## 2. `multiprocessing.Pool` — vary worker count

Real timings across 1, 2, 4, 8, and 16 worker processes on the same data and function.

In [5]:
mp_timings = {}
for procs in [1, 2, 4, 8, 16]:
    t0 = time.perf_counter()
    with mp.Pool(processes=procs) as pool:
        result_mp = pool.map(cpu_bound_transform, data, chunksize=2000)
    elapsed = time.perf_counter() - t0
    mp_timings[procs] = elapsed
    assert abs(sum(result_mp) - sum(result_single)) < 1e-6, "mismatch vs single-threaded result"
    speedup = t_single / elapsed
    print(f"multiprocessing.Pool({procs:>2} procs): {elapsed:.3f}s   speedup vs single-threaded: {speedup:.2f}x")

multiprocessing.Pool( 1 procs): 4.181s   speedup vs single-threaded: 1.04x


multiprocessing.Pool( 2 procs): 1.893s   speedup vs single-threaded: 2.29x


multiprocessing.Pool( 4 procs): 0.963s   speedup vs single-threaded: 4.50x


multiprocessing.Pool( 8 procs): 0.626s   speedup vs single-threaded: 6.92x


multiprocessing.Pool(16 procs): 0.495s   speedup vs single-threaded: 8.74x


## 3. PySpark (forward reference — not run here)

PySpark is introduced in `02-pyspark-local-mode` (not yet installed at this point in the
curriculum). It is included in this comparison conceptually, not by execution:

- `multiprocessing.Pool` parallelizes across cores **on one machine**, sharing one machine's RAM.
  If `data` does not fit in memory, or the computation needs a `groupBy`/`join` (a shuffle),
  hand-rolled multiprocessing does not help — there's no mechanism for exchanging partial
  results across partitions, spilling to disk, or recovering a lost worker's output.
- PySpark's `local[*]` mode (Task 2) runs the *same kind* of embarrassingly-parallel map across
  local cores — so for this exact toy computation, its speedup profile will look similar to
  `multiprocessing.Pool`'s, and Task 2 measures that directly. Its actual advantage shows up on
  operations `multiprocessing.Pool` cannot express at all: automatic partitioning across a
  dataset bigger than RAM, `groupBy`/`join` with shuffling handled by the engine, and
  recomputation from lineage when a worker fails mid-job.

## Overhead demo: multiprocessing on *cheap* per-row work

Hypothesis 2 above: when the per-row work is too small, `Pool` should be **slower** than a
plain loop. Real measurement, same data, a trivial function instead of the CPU-heavy one.

In [6]:
def trivial_transform(x):
    return x + 1.0

t0 = time.perf_counter()
result_single_trivial = [trivial_transform(x) for x in data]
t_single_trivial = time.perf_counter() - t0

t0 = time.perf_counter()
with mp.Pool(processes=8) as pool:
    result_mp_trivial = pool.map(trivial_transform, data, chunksize=2000)
t_mp_trivial = time.perf_counter() - t0

print(f"Single-threaded (trivial op):        {t_single_trivial:.4f}s")
print(f"multiprocessing.Pool(8) (trivial op): {t_mp_trivial:.4f}s")
print(f"multiprocessing is {t_mp_trivial / t_single_trivial:.2f}x SLOWER here (overhead dominates)")

Single-threaded (trivial op):        0.0092s
multiprocessing.Pool(8) (trivial op): 0.0561s
multiprocessing is 6.09x SLOWER here (overhead dominates)


## Shuffle cost: a real measurement, not just an assertion

Simulate why a `groupBy`/`join` ("shuffle") is expensive: data must move between partitions to
collocate rows that share a key. Compare **naive shuffle** (move every row to its key's owning
partition) against **map-side combine** (pre-aggregate locally first, only ship the small
partial aggregates) — the technique real engines (and PySpark's `groupBy` under the hood) use to
cut shuffle volume.

In [7]:
import struct

N_PARTITIONS = 8
N_KEYS = 100  # groupBy key cardinality

random.seed(7)
keyed_rows = [(random.randrange(N_KEYS), random.uniform(0, 1000)) for _ in range(N_ROWS)]

# Assign each row to a partition (as if it arrived pre-partitioned across 8 machines/cores,
# NOT by key — e.g. arrival order/hash-of-row-id), independent of its groupBy key.
partitions = [[] for _ in range(N_PARTITIONS)]
for i, row in enumerate(keyed_rows):
    partitions[i % N_PARTITIONS].append(row)

ROW_BYTES = struct.calcsize("id")  # 1 int key (8) + 1 double value (8) = 16 bytes/row, real struct size

# Naive shuffle: every row must move from its arrival partition to the partition that "owns"
# its key (key % N_PARTITIONS) before the groupBy can happen locally.
naive_shuffle_rows = 0
for p_idx, part in enumerate(partitions):
    for key, _ in part:
        owner = key % N_PARTITIONS
        if owner != p_idx:
            naive_shuffle_rows += 1
naive_shuffle_bytes = naive_shuffle_rows * ROW_BYTES

# Map-side combine: each partition pre-aggregates locally (sum, count) per key it holds, THEN
# only the small (key, partial_sum, partial_count) records are shipped to the owning partition.
combine_records = 0
for p_idx, part in enumerate(partitions):
    local_keys = set()
    for key, _ in part:
        local_keys.add(key)
    for key in local_keys:
        owner = key % N_PARTITIONS
        if owner != p_idx:
            combine_records += 1
PARTIAL_RECORD_BYTES = struct.calcsize("idi")  # key(8) + partial_sum(8) + partial_count(8) = 24 bytes
combine_bytes = combine_records * PARTIAL_RECORD_BYTES

print(f"Rows: {N_ROWS:,} across {N_PARTITIONS} partitions, {N_KEYS} distinct groupBy keys")
print(f"Naive shuffle:        {naive_shuffle_rows:,} rows moved  = {naive_shuffle_bytes / 1e6:.2f} MB")
print(f"Map-side combine:     {combine_records:,} partial records moved = {combine_bytes / 1e6:.4f} MB")
print(f"Reduction from combining before shuffling: {naive_shuffle_bytes / combine_bytes:.0f}x less data moved")

Rows: 300,000 across 8 partitions, 100 distinct groupBy keys
Naive shuffle:        262,607 rows moved  = 4.20 MB
Map-side combine:     700 partial records moved = 0.0140 MB
Reduction from combining before shuffling: 300x less data moved


## Fault tolerance demo: recompute from lineage vs replicate

A real (small) demonstration of the two strategies for surviving a lost partition's result:

- **Replication**: keep a second copy of the *result* somewhere else. Safe, but doubles storage
  and doesn't help if the computation itself needs to change.
- **Lineage (recomputation)**: don't store the result redundantly — store the *recipe*
  (source partition + the transform function) and recompute it if lost. This is what Spark's
  RDD lineage graph does.

In [8]:
def compute_partition(partition_id, source_partition):
    """The 'recipe': deterministic transform applied to one partition."""
    return [cpu_bound_transform(x) for x in source_partition]

# Simulate 4 partitions, each with its own lineage (partition id + source data + the function).
source_partitions = [data[i::4] for i in range(4)]
lineage = {pid: (compute_partition, pid, source_partitions[pid]) for pid in range(4)}

# Compute all results the first time.
results = {}
for pid, (fn, _, src) in lineage.items():
    results[pid] = fn(pid, src)
print("Initial computation done. Partition sizes:", {pid: len(r) for pid, r in results.items()})

# Simulate "losing" partition 2's result (e.g. that worker machine crashed).
del results[2]
print("Simulated failure: partition 2's result lost. Results now held for partitions:", list(results.keys()))

# Recover using lineage: recompute ONLY the lost partition, from its recorded recipe.
fn, pid, src = lineage[2]
t0 = time.perf_counter()
results[2] = fn(pid, src)
t_recompute = time.perf_counter() - t0
print(f"Recovered partition 2 by recomputation in {t_recompute:.3f}s (only that partition, not the whole job)")
print("Recovered result matches original transform:",
      results[2] == [cpu_bound_transform(x) for x in source_partitions[2]])

Initial computation done. Partition sizes: {0: 75000, 1: 75000, 2: 75000, 3: 75000}
Simulated failure: partition 2's result lost. Results now held for partitions: [0, 1, 3]


Recovered partition 2 by recomputation in 0.944s (only that partition, not the whole job)


Recovered result matches original transform: True


## Summary table

In [9]:
print(f"{'Approach':35s} {'Time (s)':>10s} {'Speedup':>10s}")
print(f"{'Single-threaded loop':35s} {t_single:10.3f} {'1.00x':>10s}")
for procs, elapsed in mp_timings.items():
    print(f"{'multiprocessing.Pool(' + str(procs) + ')':35s} {elapsed:10.3f} {t_single/elapsed:9.2f}x")

Approach                              Time (s)    Speedup
Single-threaded loop                     4.331      1.00x
multiprocessing.Pool(1)                  4.181      1.04x
multiprocessing.Pool(2)                  1.893      2.29x
multiprocessing.Pool(4)                  0.963      4.50x
multiprocessing.Pool(8)                  0.626      6.92x
multiprocessing.Pool(16)                 0.495      8.74x
